# US Traffic Fatality Prediction — Data Preparation
**DATA 495: Data Science Capstone**
**Carl Stolpe | UMGC | May 2026** (data-coding fixes November 2026)

This notebook executes the full data preparation pipeline: sentinel code remapping, feature engineering, temporal train/validation splitting, and imputation. All transformations are fit on the 2015 training set only and applied consistently to the 2016 validation set to prevent data leakage.

**What the model predicts:** FARS is a census of *fatal* crashes only, so every record here comes from a crash in which at least one person died. The target is whether **this driver** was fatally injured, given that they were in a fatal crash.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## 2. Load and Join Raw FARS Tables

In [2]:
def load_and_join(year):
    """Load accident and person CSVs, join on ST_CASE+YEAR, filter to drivers."""
    acc = pd.read_csv(f'../data/raw/accident_{year}.csv', low_memory=False, encoding='latin1')
    per = pd.read_csv(f'../data/raw/person_{year}.csv',   low_memory=False, encoding='latin1')
    acc['YEAR'] = year
    per['YEAR'] = year
    # The person file repeats many crash-level columns; take only the
    # accident-only ones to avoid HOUR_x / HOUR_y duplicates after merging.
    acc_cols = ['ST_CASE', 'YEAR'] + [c for c in acc.columns if c not in per.columns]
    merged  = per.merge(acc[acc_cols], on=['ST_CASE', 'YEAR'], how='inner')
    drivers = merged[merged['PER_TYP'] == 1].copy()
    print(f'{year}: {len(drivers):,} driver records')
    return drivers

df_2015 = load_and_join(2015)
df_2016 = load_and_join(2016)

2015: 49,163 driver records


2016: 52,399 driver records


## 3. Binary Target Variable

In [3]:
def recode_target(df):
    """Recode INJ_SEV to binary: fatal (4) = 1, other known levels = 0.

    Drops INJ_SEV 6 (died prior to crash) and 9 (unknown) instead of
    counting them as non-fatal.
    """
    df = df[~df['INJ_SEV'].isin([6, 9])].copy()
    df['INJ_SEV_BINARY'] = (df['INJ_SEV'] == 4).astype(int)
    return df

df_2015 = recode_target(df_2015)
df_2016 = recode_target(df_2016)

print(f'2015: {len(df_2015):,} drivers, fatal rate {df_2015["INJ_SEV_BINARY"].mean():.1%}')
print(f'2016: {len(df_2016):,} drivers, fatal rate {df_2016["INJ_SEV_BINARY"].mean():.1%}')

2015: 48,533 drivers, fatal rate 46.1%
2016: 51,195 drivers, fatal rate 46.3%


## 4. Sentinel Code Remapping

Sentinel codes representing unknown or inapplicable values are remapped to NaN before any imputation is applied.

In [4]:
# FARS 2015-2016 codes for unknown / not reported / not applicable
SENTINELS = {
    'HOUR':     [99],
    'AGE':      [998, 999],
    'SEX':      [8, 9],
    'MOD_YEAR': [9998, 9999],
    'LGT_COND': [8, 9],
    'WEATHER':  [98, 99],
    'MAN_COLL': [98, 99],
    'FUNC_SYS': [96, 98, 99],
    'RUR_URB':  [6, 8, 9],
    'REST_USE': [96, 97, 98, 99],
}

def remap_sentinels(df):
    """Replace FARS sentinel codes with NaN."""
    df = df.copy()
    for col, codes in SENTINELS.items():
        if col in df.columns:
            df[col] = df[col].replace(codes, np.nan)
    return df

df_2015 = remap_sentinels(df_2015)
df_2016 = remap_sentinels(df_2016)
print('Sentinel codes remapped.')

Sentinel codes remapped.


## 5. Feature Engineering

- **Alcohol.** `ALC_RES` is stored in thousandths of g/dL (`80` = 0.080), with 995–999 as status codes (not reported, test not given, result unknown, positive without value, unknown). `ALC_POSITIVE` = 1 when the measured BAC is 0.08 or higher (`80 <= ALC_RES <= 940`). `ALC_TESTED` is kept for reference but not used as a model feature, because fatally injured drivers are tested much more often than survivors — it would leak the outcome.
- **Rural/urban.** Taken from the accident file's `RUR_URB` field (1 = rural, 2 = urban). `FUNC_SYS` is the road's functional class (Interstate, arterial, collector, local), not a rural/urban flag, and stays a separate feature.

In [5]:
def engineer_features(df):
    """Engineer ALC_TESTED, ALC_POSITIVE, and RURAL_URBAN features."""
    df = df.copy()

    # Alcohol features — ALC_RES is in thousandths (80 = 0.080 BAC);
    # 995-999 are status codes, not test results.
    alc = df['ALC_RES']
    measured = alc.notna() & (alc <= 940)
    if measured.any() and alc[measured].max() < 2:
        raise ValueError('ALC_RES looks like decimal BAC, not thousandths — '
                         'check which FARS file format you downloaded.')
    df['ALC_TESTED']   = (measured | alc.isin([997, 998])).astype(int)
    df['ALC_POSITIVE'] = (measured & (alc >= 80)).astype(int)

    # Rural/urban flag from RUR_URB (1 = rural, 2 = urban; unknowns already NaN)
    df['RURAL_URBAN'] = df['RUR_URB'].map({1: 1, 2: 0})

    return df

df_2015 = engineer_features(df_2015)
df_2016 = engineer_features(df_2016)
print('Features engineered.')
for name, d in [('2015', df_2015), ('2016', df_2016)]:
    print(f'  {name}: tested {d["ALC_TESTED"].mean():.1%}, '
          f'BAC>=0.08 {d["ALC_POSITIVE"].mean():.1%}, '
          f'rural {d["RURAL_URBAN"].mean():.1%}')

Features engineered.
  2015: tested 51.3%, BAC>=0.08 13.4%, rural 48.6%
  2016: tested 49.4%, BAC>=0.08 13.1%, rural 47.3%


## 6. Select Final Feature Set

`FATALS` (number of deaths in the crash) and `ALC_TESTED` are saved for descriptive use but are **not** model features: both are only known after the outcome and would leak it (for example, in a single-vehicle crash with one death, that death is almost always the driver).

In [6]:
NUMERIC_FEATURES = ['AGE', 'HOUR', 'MONTH', 'DRUNK_DR']
CATEGORICAL_FEATURES = [
    'SEX', 'LGT_COND', 'WEATHER', 'MAN_COLL', 'FUNC_SYS',
    'RURAL_URBAN', 'STATE', 'REST_USE', 'ALC_POSITIVE'
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
EXTRA_COLS = ['FATALS', 'ALC_TESTED']   # descriptive only, not model inputs (HOUR_RAW added below)
TARGET = 'INJ_SEV_BINARY'

train_raw = df_2015[FEATURES + EXTRA_COLS + [TARGET]].copy()
val_raw   = df_2016[FEATURES + EXTRA_COLS + [TARGET]].copy()

# Keep the un-imputed hour for charts (imputed hours would all land on the median)
train_raw['HOUR_RAW'] = train_raw['HOUR']
val_raw['HOUR_RAW']   = val_raw['HOUR']

print(f'Training features:   {train_raw.shape}')
print(f'Validation features: {val_raw.shape}')
print(f'\nMissing values — Training:')
missing = train_raw[FEATURES].isnull().sum()
print(missing[missing > 0])

Training features:   (48533, 17)
Validation features: (51195, 17)

Missing values — Training:
AGE             496
HOUR            234
SEX             469
LGT_COND        156
WEATHER         326
MAN_COLL        129
FUNC_SYS        360
RURAL_URBAN     258
REST_USE       3760
dtype: int64


## 7. Imputation

- **Numeric features** (age, hour, month, drunk-driver count): missing values are filled with the **2015 training median**, applied to both years to prevent leakage.
- **Categorical features** (codes such as weather, restraint use, state): missing values become their own category, `-1` ("unknown"). Filling a category code with a median would invent a real-looking category, and "unknown" can itself be informative.

In [7]:
# Numeric: medians from the training set only
train_medians = train_raw[NUMERIC_FEATURES].median()

train_clean = train_raw.copy()
val_clean   = val_raw.copy()

train_clean[NUMERIC_FEATURES] = train_clean[NUMERIC_FEATURES].fillna(train_medians)
val_clean[NUMERIC_FEATURES]   = val_clean[NUMERIC_FEATURES].fillna(train_medians)

# Categorical: missing -> -1 ("unknown") category
train_clean[CATEGORICAL_FEATURES] = train_clean[CATEGORICAL_FEATURES].fillna(-1).astype(int)
val_clean[CATEGORICAL_FEATURES]   = val_clean[CATEGORICAL_FEATURES].fillna(-1).astype(int)

# Confirm no remaining nulls in model features
assert train_clean[FEATURES].isnull().sum().sum() == 0, 'Training NaNs remain'
assert val_clean[FEATURES].isnull().sum().sum()   == 0, 'Validation NaNs remain'

print('Imputation complete. No remaining missing values in model features.')
print(f'\nTraining medians used for numeric imputation:')
print(train_medians.round(2))
print(f'\nShare coded as unknown (-1) — Training:')
print((train_clean[CATEGORICAL_FEATURES] == -1).mean().round(3))

Imputation complete. No remaining missing values in model features.



Training medians used for numeric imputation:
AGE         40.0
HOUR        14.0
MONTH        7.0
DRUNK_DR     0.0
dtype: float64

Share coded as unknown (-1) — Training:
SEX             0.010
LGT_COND        0.003
WEATHER         0.007
MAN_COLL        0.003
FUNC_SYS        0.007
RURAL_URBAN     0.005
STATE           0.000
REST_USE        0.077
ALC_POSITIVE    0.000
dtype: float64


## 8. Save Prepared Datasets

In [8]:
import os
os.makedirs('../data', exist_ok=True)

train_clean.to_csv('../data/fars_2015_prepared.csv', index=False)
val_clean.to_csv('../data/fars_2016_prepared.csv',   index=False)

print('Prepared datasets saved.')
print(f'  fars_2015_prepared.csv — {len(train_clean):,} records, {train_clean.shape[1]} columns')
print(f'  fars_2016_prepared.csv — {len(val_clean):,} records, {val_clean.shape[1]} columns')

Prepared datasets saved.
  fars_2015_prepared.csv — 48,533 records, 17 columns
  fars_2016_prepared.csv — 51,195 records, 17 columns


## 9. Final Dataset Summary

In [9]:
print('Final Prepared Dataset Summary')
print('=' * 50)
print(f'Training set (2015):   {len(train_clean):,} records')
print(f'Validation set (2016): {len(val_clean):,} records')
print(f'Total:                 {len(train_clean)+len(val_clean):,} records')
print(f'Model features:        {len(FEATURES)}')
print(f'Target:                {TARGET} (binary)')
print(f'Training fatal rate:   {train_clean[TARGET].mean():.1%}')
print(f'Validation fatal rate: {val_clean[TARGET].mean():.1%}')
print(f'Missing values:        0 in model features (post-imputation)')
print(f'Split method:          Temporal (2015=train / 2016=validation)')

Final Prepared Dataset Summary
Training set (2015):   48,533 records
Validation set (2016): 51,195 records
Total:                 99,728 records
Model features:        13
Target:                INJ_SEV_BINARY (binary)
Training fatal rate:   46.1%
Validation fatal rate: 46.3%
Missing values:        0 in model features (post-imputation)
Split method:          Temporal (2015=train / 2016=validation)
